# M1 Dispatch Priority Weight Sensitivity Audit

28번 priority 가중치를 확정값이 아니라 정책 후보로 두고, 여러 가중치에서 순위가 얼마나 흔들리는지 확인한다.

## Context & Methods

- 새 ML 모델은 학습하지 않는다.
- 28번 pre-event risk, 29번 stable lead-time, 28번 group weight를 그대로 사용한다.
- 목적은 `why`를 설명할 수 있는 priority policy 후보를 고르는 것이다.

In [1]:
from scripts.run_30_dispatch_priority_weight_sensitivity_audit import run_analysis
results = run_analysis(write_notebook_file=False)
results['decision_matrix']

30 M1 dispatch priority weight sensitivity audit complete
       scenario  top10_overlap_rate  high_tier_ratio review_events_in_top10  passes_all_guardrails scenario_decision                final_decision
    baseline_28                 1.0         0.636364                                          True       recommended baseline_28_keep_as_policy_v1
     risk_heavy                 0.7         0.696970                     67                  False  reject_or_review baseline_28_keep_as_policy_v1
 leadtime_heavy                 1.0         0.666667                                          True         candidate baseline_28_keep_as_policy_v1
    group_heavy                 0.7         0.545455                                          True         candidate baseline_28_keep_as_policy_v1
balanced_policy                 1.0         0.636364                                          True         candidate baseline_28_keep_as_policy_v1
                                                   check  pa

,scenario,w_risk,w_leadtime,w_group,policy_intent,weight_sum,formula,top5_overlap_count,top5_overlap_rate,top10_overlap_count,...,passes_review_guardrail,passes_high_tier_ratio,passes_long_leadtime_guardrail,passes_low_risk_guardrail,passes_all_guardrails,guardrail_pass_count,decision_rationale,scenario_decision,recommended_scenario,final_decision
0,baseline_28,0.55,0.30,0.15,28번 기준선: risk를 가장 크게 보고 leadtime과 group을 보조로 사용,1.0,100*(0.55*risk_probability+0.30*leadtime_urgen...,5,1.0,10,...,True,True,True,True,True,5,all_guardrails_pass,recommended,baseline_28,baseline_28_keep_as_policy_v1
1,risk_heavy,0.70,0.20,0.10,현재 모델 위험확률을 더 강하게 반영,1.0,100*(0.70*risk_probability+0.20*leadtime_urgen...,3,0.6,7,...,False,True,True,True,False,4,review_event_enters_top10,reject_or_review,baseline_28,baseline_28_keep_as_policy_v1
2,leadtime_heavy,0.45,0.40,0.15,며칠 전부터 잡히는 신호와 출동 여유를 더 강하게 반영,1.0,100*(0.45*risk_probability+0.40*leadtime_urgen...,4,0.8,10,...,True,True,True,True,True,5,all_guardrails_pass,candidate,baseline_28,baseline_28_keep_as_policy_v1
3,group_heavy,0.45,0.25,0.30,빈도와 monitoring potential이 높은 고장군을 더 강하게 반영,1.0,100*(0.45*risk_probability+0.25*leadtime_urgen...,4,0.8,7,...,True,True,True,True,True,5,all_guardrails_pass,candidate,baseline_28,baseline_28_keep_as_policy_v1
4,balanced_policy,0.50,0.30,0.20,risk 중심은 유지하되 group 정보를 baseline보다 조금 더 반영,1.0,100*(0.50*risk_probability+0.30*leadtime_urgen...,4,0.8,10,...,True,True,True,True,True,5,all_guardrails_pass,candidate,baseline_28,baseline_28_keep_as_policy_v1


## Data

가중치 시나리오와 event별 score를 확인한다.

In [2]:
display(results['scenarios'])
display(results['scores'].head(15))

,scenario,w_risk,w_leadtime,w_group,policy_intent,weight_sum,formula
0,baseline_28,0.55,0.30,0.15,28번 기준선: risk를 가장 크게 보고 leadtime과 group을 보조로 사용,1.0,100*(0.55*risk_probability+0.30*leadtime_urgen...
1,risk_heavy,0.70,0.20,0.10,현재 모델 위험확률을 더 강하게 반영,1.0,100*(0.70*risk_probability+0.20*leadtime_urgen...
2,leadtime_heavy,0.45,0.40,0.15,며칠 전부터 잡히는 신호와 출동 여유를 더 강하게 반영,1.0,100*(0.45*risk_probability+0.40*leadtime_urgen...
3,group_heavy,0.45,0.25,0.30,빈도와 monitoring potential이 높은 고장군을 더 강하게 반영,1.0,100*(0.45*risk_probability+0.25*leadtime_urgen...
4,balanced_policy,0.50,0.30,0.20,risk 중심은 유지하되 group 정보를 baseline보다 조금 더 반영,1.0,100*(0.50*risk_probability+0.30*leadtime_urgen...


,event_id,substation_id,fault_label,problem_en,efd_possible,monitoring_potential,report_date,first_crossing_lead_time_hours,stable_crossing_lead_time_hours,d0_probability,...,w_leadtime,w_group,scenario_score,scenario_tier,scenario_rank,baseline_rank,baseline_score,rank_change_from_baseline,abs_rank_change_from_baseline,score_change_from_baseline
0,3,12,Control unit: Incorrect parameterisation,no heat,True,4.0,2015-12-01 10:56:00,168.0,0.0,0.985404,...,0.3,0.15,99.197210,high,1,1,99.197210,0,0,0.0
1,49,18,Control unit: Incorrect parameterisation,no DHW,True,4.0,2019-05-04 07:19:00,0.0,0.0,0.960765,...,0.3,0.15,97.842049,high,2,2,97.842049,0,0,0.0
2,60,4,Control unit: Incorrect parameterisation,no heat,False,4.0,2015-03-18 18:54:00,72.0,0.0,0.911346,...,0.3,0.15,95.124004,high,3,3,95.124004,0,0,0.0
3,23,27,Control unit: Incorrect parameterisation,not enough heat,True,4.0,2020-01-20 12:57:00,72.0,0.0,0.898881,...,0.3,0.15,94.438460,high,4,4,94.438460,0,0,0.0
4,5,11,Failure of the heating circuit pump,no heat,True,3.8,2018-11-23 08:30:00,24.0,24.0,0.983612,...,0.3,0.15,93.639300,high,5,5,93.639300,0,0,0.0
5,53,13,Shut-off valve closed,no DHW,True,2.8,2020-05-26 10:00:00,168.0,0.0,0.991303,...,0.3,0.15,93.235083,high,6,6,93.235083,0,0,0.0
6,62,21,Control unit: Incorrect parameterisation,not enough heat,True,4.0,2019-01-21 08:17:00,24.0,0.0,0.876529,...,0.3,0.15,93.209114,high,7,7,93.209114,0,0,0.0
7,7,26,Incorrect setting of the differential pressure...,no DHW,True,3.1,2020-06-13 10:38:00,168.0,0.0,0.999994,...,0.3,0.15,92.825216,high,8,8,92.825216,0,0,0.0
8,32,21,Control unit: Incorrect parameterisation,not enough heat,True,4.0,2019-10-19 10:38:00,168.0,0.0,0.860889,...,0.3,0.15,92.348893,high,9,9,92.348893,0,0,0.0
9,44,8,Temperature monitor/controller defective,no DHW,True,3.0,2018-04-27 08:15:00,12.0,12.0,0.858773,...,0.3,0.15,92.232504,high,10,10,92.232504,0,0,0.0


## Results

상위 10개 안정성, rank 변화, tier 분포를 확인한다.

In [3]:
display(results['rank_changes'])
display(results['group_summary'].head(20))
display(results['quality_audit'])

,scenario,top5_overlap_count,top5_overlap_rate,top10_overlap_count,top10_overlap_rate,mean_abs_rank_change,max_abs_rank_change,high_count,medium_count,low_count,...,review_events_in_top10,review_top10_count,long_leadtime_high_or_medium_count,low_risk_high_count,passes_top10_overlap,passes_review_guardrail,passes_high_tier_ratio,passes_long_leadtime_guardrail,passes_low_risk_guardrail,passes_all_guardrails
0,baseline_28,5,1.0,10,1.0,0.000000,0,21,4,0,...,,0,6,0,True,True,True,True,True,True
1,risk_heavy,3,0.6,7,0.7,2.242424,5,23,2,0,...,67,1,6,0,True,False,True,True,True,False
2,leadtime_heavy,4,0.8,10,1.0,0.666667,3,22,2,1,...,,0,5,0,True,True,True,True,True,True
3,group_heavy,4,0.8,7,0.7,2.181818,8,18,6,1,...,,0,5,0,True,True,True,True,True,True
4,balanced_policy,4,0.8,10,1.0,1.030303,4,21,4,0,...,,0,6,0,True,True,True,True,True,True


,scenario,fault_group,leadtime_label,leadtime_confidence,event_count,mean_score,median_score,min_rank,median_rank,high_count,medium_count,low_count,monitor_count,high_or_medium_ratio
0,balanced_policy,pressure_regulator,report_time_only,high,4,88.212813,88.705873,10,14.5,4,0,0,0,1.000000
1,balanced_policy,control_controller,report_time_only,high,12,85.172852,92.991544,1,6.5,10,0,0,2,0.833333
2,balanced_policy,pump_failure,short_stable,high,5,75.740217,81.045294,8,21.0,3,1,0,1,0.800000
3,balanced_policy,valve_actuator,early_stable,medium,5,58.472880,73.606013,9,23.0,2,1,0,2,0.600000
4,balanced_policy,leakage_water_loss,early_stable,high,5,55.795147,66.123492,20,25.0,1,2,0,2,0.600000
5,balanced_policy,unknown_review,review_only,review,2,51.489067,51.489067,19,25.5,1,0,0,1,0.500000
6,baseline_28,pressure_regulator,report_time_only,high,4,90.382180,90.924545,8,12.5,4,0,0,0,1.000000
7,baseline_28,control_controller,report_time_only,high,12,84.190137,92.290698,1,9.5,10,0,0,2,0.833333
8,baseline_28,pump_failure,short_stable,high,5,76.641941,82.597526,5,21.0,3,1,0,1,0.800000
9,baseline_28,valve_actuator,early_stable,medium,5,58.813913,75.700359,6,23.0,2,1,0,2,0.600000


,check,pass,detail
0,input_exists_m1_fault_dispatch_priority_v1.csv,True,C:\3rd_Project\HeatGridAgent\07_데이터산출물\m1_faul...
1,input_exists_m1_fault_group_leadtime_summary.csv,True,C:\3rd_Project\HeatGridAgent\07_데이터산출물\m1_faul...
2,input_exists_m1_fault_group_leadtime_decision_...,True,C:\3rd_Project\HeatGridAgent\07_데이터산출물\m1_faul...
3,input_exists_m1_fault_pre_event_v1_lock_decisi...,True,C:\3rd_Project\HeatGridAgent\07_데이터산출물\m1_faul...
4,pre_event_lock_unchanged,True,decision=fault_pre_event_gate_v1_locked_for_M1...
5,no_new_model_training,True,priority sensitivity uses existing 28/29 outpu...
6,all_weight_sums_equal_one,True,baseline_28:1.000000|risk_heavy:1.000000|leadt...
7,baseline_score_matches_28_priority,True,max_diff=2.842170943040401e-14
8,stable_leadtime_label_used,True,29 leadtime decision matrix joined by fault_group
9,special_event_flags_retained,True,20|34|67|69


## Takeaways

- 28번 가중치는 절대 정답이 아니라 baseline policy candidate다.
- 가중치 선택 근거는 top10 안정성, review event guardrail, high tier 비율이다.
- 외부 출동 비용 데이터가 생기기 전까지는 가중치 학습보다 민감도 audit이 더 안전하다.